# 5. Evaluation: From Gut Checks to Automated Judgment

### Setup

In [10]:
import sys 
sys.path.insert(0, "..") 
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base 
import requests 
import json 
from datetime import datetime

# connect to the chromadb
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(name="rpg_rules")

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("../models/granite-embedding-30m-english")


print(f"✅ Connected to ChromaDB — {collection.count()} chunks loaded")



✅ Connected to ChromaDB — 1143 chunks loaded


/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


## 5.1 The Manual Phase: Ask Questions, Write Down What's Wrong

This is how every customer starts. Someone sits in front of the system, asks a few questions, and forms an opinion. That opinion usually sounds like one of these:

* "It kind of works."
* "Some answers are good, some are weird."
* "I don't trust it yet."

We're going to do the same thing, deliberately. But unlike the customer, we're going to be structured about it. We already have two questions from earlier in the lab. Now we're going to expand that to a small set, run each question two ways, without context and with RAG,  and record what we observe.

This isn't sophisticated. It's not meant to be. The point is to do what the customer does, feel why it breaks down, and understand what we'd need to do it better.


In [6]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "known_answer": "The Thief may only try once per lock. If they fail, they must wait until they gain another level of experience before trying again.",
        "why": "Control question — we already verified this in Section 4."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "known_answer": "Elves use a d6 for hit dice because of their character class restrictions in Basic Fantasy RPG.",
        "why": "Tests whether the model answers from this game or defaults to D&D."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "known_answer": "No. Only Magic-Users (and in some cases Thieves at higher levels) can use magic-user scrolls.",
        "why": "Requires interpreting class restriction rules — not a single sentence answer."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "known_answer": "Each side rolls 1d6 at the start of each round. The side with the higher roll acts first.",
        "why": "A rule that differs significantly across RPG systems. Tests domain specificity."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "known_answer": "This depends on the character's Charisma score. The Charisma table defines the maximum number of retainers.",
        "why": "Answer requires connecting two pieces of information — Charisma rules and retainer rules."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions loaded")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

Evaluation set: 5 questions loaded
  [q1] What happens if a Thief fails an Open Locks attempt?
  [q2] Why can't Elves roll higher than a d6 for hit points?
  [q3] Can a Fighter use magic-user scrolls?
  [q4] How is initiative determined in combat?
  [q5] What is the maximum number of retainers a character can hire?


In [8]:
url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

def ask_without_context(question, model=model_id):
    """Baseline: model answers from its own training data only."""
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {e}"


def ask_with_rag(question, collection, model=model_id, n_results=3):
    """RAG: retrieve relevant chunks, then ask the model."""
    # Retrieve (embed the question manually)
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    context_chunks = results["documents"][0]
    context = "\n\n---\n\n".join(context_chunks)

    # Build prompt
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer, context_chunks
    except Exception as e:
        return f"ERROR: {e}", context_chunks

In [11]:
results = []

print("Running evaluation set...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")

    # Baseline (no context)
    baseline_answer = ask_without_context(q["question"])

    # RAG (with retrieved context)
    rag_answer, retrieved_chunks = ask_with_rag(q["question"], collection)

    result = {
        "id": q["id"],
        "question": q["question"],
        "known_answer": q["known_answer"],
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer,
        "retrieved_chunks": retrieved_chunks,
        "timestamp": datetime.now().isoformat()
    }
    results.append(result)
    print(f"  ✅ Done\n")

print(f"All {len(results)} questions complete.")


Running evaluation set...

[q1] What happens if a Thief fails an Open Locks attempt?
  ✅ Done

[q2] Why can't Elves roll higher than a d6 for hit points?
  ✅ Done

[q3] Can a Fighter use magic-user scrolls?
  ✅ Done

[q4] How is initiative determined in combat?
  ✅ Done

[q5] What is the maximum number of retainers a character can hire?
  ✅ Done

All 5 questions complete.


In [12]:
print("=" * 80)
print("EVALUATION RESULTS: Baseline vs. RAG")
print("=" * 80)

for r in results:
    print(f"\n{'─' * 80}")
    print(f"Q{r['id']}: {r['question']}")
    print(f"{'─' * 80}")
    print(f"\n🧠 Baseline (no context):\n{r['baseline_answer']}")
    print(f"\n📚 RAG (with retrieved context):\n{r['rag_answer']}")
    print(f"\n📎 Retrieved chunks: {len(r['retrieved_chunks'])}")

EVALUATION RESULTS: Baseline vs. RAG

────────────────────────────────────────────────────────────────────────────────
Qq1: What happens if a Thief fails an Open Locks attempt?
────────────────────────────────────────────────────────────────────────────────

🧠 Baseline (no context):
If a Thief fails an Open Locks attempt, they do not make any progress towards opening the lock. The Thief must wait until their next attempt to try again. If the Thief fails the roll by 4 or more, they may trigger a trap or alert nearby enemies.

📚 RAG (with retrieved context):
If a Thief fails an Open Locks attempt, they cannot unlock the lock without a proper key. Furthermore, they must wait until they have gained another level of experience before trying again.

📎 Retrieved chunks: 3

────────────────────────────────────────────────────────────────────────────────
Qq2: Why can't Elves roll higher than a d6 for hit points?
────────────────────────────────────────────────────────────────────────────────

🧠

In [13]:
# Simple manual scoring framework
print("SCORING GUIDE")
print("  0 = Wrong or hallucinated")
print("  1 = Partially correct or vague")
print("  2 = Correct and grounded\n")

for r in results:
    print(f"Q{r['id']}: {r['question']}")
    print(f"  Baseline: ___")
    print(f"  RAG:      ___")
    print()

SCORING GUIDE
  0 = Wrong or hallucinated
  1 = Partially correct or vague
  2 = Correct and grounded

Qq1: What happens if a Thief fails an Open Locks attempt?
  Baseline: ___
  RAG:      ___

Qq2: Why can't Elves roll higher than a d6 for hit points?
  Baseline: ___
  RAG:      ___

Qq3: Can a Fighter use magic-user scrolls?
  Baseline: ___
  RAG:      ___

Qq4: How is initiative determined in combat?
  Baseline: ___
  RAG:      ___

Qq5: What is the maximum number of retainers a character can hire?
  Baseline: ___
  RAG:      ___



In [16]:
import ipywidgets as widgets
from IPython.display import display, HTML

scores = {}

def create_evaluation_ui(results):
    tabs = []
    
    for r in results:
        q_id = r["id"]
        
        # Display content
        content = widgets.HTML(value=f"""
        <h3>Q{q_id}: {r['question']}</h3>
        <h4>🧠 Baseline (no context):</h4>
        <p style="background:rgba(128,128,128,0.15); padding:10px; border-radius:5px;">{r['baseline_answer']}</p>
        <h4>📚 RAG (with retrieved context):</h4>
        <p style="background:rgba(76,175,80,0.15); padding:10px; border-radius:5px;">{r['rag_answer']}</p>
        """)
        
        # Scoring dropdowns
        baseline_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="Baseline:",
            style={"description_width": "80px"}
        )
        rag_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="RAG:",
            style={"description_width": "80px"}
        )
        
        notes = widgets.Textarea(
            placeholder="Optional notes — why did you score it this way?",
            layout=widgets.Layout(width="100%", height="60px")
        )
        
        # Save scores on change
        def make_handler(qid, field):
            def handler(change):
                if qid not in scores:
                    scores[qid] = {}
                scores[qid][field] = change["new"]
            return handler
        
        baseline_score.observe(make_handler(q_id, "baseline"), names="value")
        rag_score.observe(make_handler(q_id, "rag"), names="value")
        notes.observe(make_handler(q_id, "notes"), names="value")
        
        tab_content = widgets.VBox([content, baseline_score, rag_score, notes])
        tabs.append(tab_content)
    
    tab_widget = widgets.Tab(children=tabs)
    for i, r in enumerate(results):
        tab_widget.set_title(i, f"Q{r['id']}")
    
    display(tab_widget)

create_evaluation_ui(results)

In [17]:
def show_scores():
    print("EVALUATION SUMMARY")
    print("=" * 50)
    print(f"{'Question':<12} {'Baseline':>10} {'RAG':>10} {'Delta':>10}")
    print("-" * 50)
    
    baseline_total = 0
    rag_total = 0
    
    for q_id, s in sorted(scores.items()):
        b = s.get("baseline", "—")
        r = s.get("rag", "—")
        delta = ""
        if isinstance(b, int) and isinstance(r, int):
            delta = f"+{r - b}" if r >= b else str(r - b)
            baseline_total += b
            rag_total += r
        print(f"Q{q_id:<11} {str(b):>10} {str(r):>10} {delta:>10}")
    
    print("-" * 50)
    print(f"{'Total':<12} {baseline_total:>10} {rag_total:>10} {f'+{rag_total - baseline_total}':>10}")

show_scores()

EVALUATION SUMMARY
Question       Baseline        RAG      Delta
--------------------------------------------------
Qq1                   0          2         +2
Qq2                   0          2         +2
--------------------------------------------------
Total                 0          4         +4


## 5.3  From Manual Checks to Structured Evaluation: Add expected answers so you can score programmatically instead of just eyeballing:


In [19]:
import ipywidgets as widgets
from IPython.display import display, HTML

scores = {}

def create_evaluation_ui(results):
    tabs = []
    
    for r in results:
        q_id = r["id"]
        
        # Display content
        content = widgets.HTML(value=f"""
        <h3>Q{q_id}: {r['question']}</h3>
        <h4>🧠 Baseline (no context):</h4>
        <p style="background:rgba(128,128,128,0.15); padding:10px; border-radius:5px;">{r['baseline_answer']}</p>
        <h4>📚 RAG (with retrieved context):</h4>
        <p style="background:rgba(76,175,80,0.15); padding:10px; border-radius:5px;">{r['rag_answer']}</p>
        <h4>📎 Retrieved Chunks:</h4>
        <details>
            <summary>Click to expand ({len(r['retrieved_chunks'])} chunks)</summary>
            {"".join(f'<p style="background:rgba(128,128,128,0.1); padding:8px; border-radius:5px; margin:4px 0; font-size:0.85em;">{chunk}</p>' for chunk in r['retrieved_chunks'])}
        </details>
        """)
        
        # Scoring dropdowns
        baseline_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="Baseline:",
            style={"description_width": "80px"}
        )
        rag_score = widgets.Dropdown(
            options=[("—", None), ("0 - Wrong/Hallucinated", 0), ("1 - Partial/Vague", 1), ("2 - Correct/Grounded", 2)],
            description="RAG:",
            style={"description_width": "80px"}
        )
        
        notes = widgets.Textarea(
            placeholder="Optional notes — why did you score it this way?",
            layout=widgets.Layout(width="100%", height="60px")
        )
        
        # Save scores on change
        def make_handler(qid, field):
            def handler(change):
                if qid not in scores:
                    scores[qid] = {}
                scores[qid][field] = change["new"]
            return handler
        
        baseline_score.observe(make_handler(q_id, "baseline"), names="value")
        rag_score.observe(make_handler(q_id, "rag"), names="value")
        notes.observe(make_handler(q_id, "notes"), names="value")
        
        tab_content = widgets.VBox([
            content,
            widgets.HBox([baseline_score, rag_score]),
            notes
        ])
        tabs.append(tab_content)
    
    tab_widget = widgets.Tab(children=tabs)
    for i, r in enumerate(results):
        tab_widget.set_title(i, f"Q{r['id']}")
    
    display(tab_widget)

create_evaluation_ui(results)